# 39차 — GBM 3사 그리드 서치: 가족별 자기 튜닝 후 공정 재대결 (MLflow)

> 배경: 04 §3b의 GBM 3사 비교는 동일 스크리닝 설정이라 **XGBoost·CatBoost는 "자기 튜닝"을 받은 적이
> 없었다** — "튜닝하면 역전하는가?"가 미검증 갭. 세션2 튜토리얼의 파라미터 조합 루프 패턴으로
> 조합별 MLflow run을 남기며 그리드 서치로 닫는다. 작성: 2026-07-29 · env: `sajura-ai`(+mlflow)

🔒 출력 제거 커밋 — 지표는 sMAPE·상대값만, `mlruns/`는 gitignore 로컬 전용(sqlite).

**사전 선언(결과 확인 전 고정)**
- 그리드: **LGBM 36 · XGB 36 · CatBoost 18 = 90조합** × 5 fold(총 450 fit) — 전부 비율 타깃 하네스,
  n_estimators 800 + 조기 종료 50(기존 프로토콜), test fold(2026-04) 봉인 유지.
- 판정: 가족별 best(선택 fold 평균 MAE)가 **V1-t 대비 −5% 이상 개선 시 격상**.
  LGBM 재탐색 best가 채택값보다 좋아도 **즉시 교체 없음** — 그리드 best 선택은 검증 fold 낙관
  편향이 있으므로 "재튜닝 검토" 기록만.

## 판정 요약 (TL;DR)

1. **3가족 전부 기각 — V1-t 유지.** 가족별 best: LGBM 재탐색 sMAPE 49.16(**vs V1-t −0.2%**, 동률) ·
   XGBoost 51.39(**+10.2%**) · CatBoost 49.99(+11.0%).
2. **"튜닝 부족" 가설 종결** — XGB·CatBoost는 자기 그리드로 튜닝해도 ~10% 열세 유지.
   가족 간 격차는 설정 문제가 아니라 실재(소표본에서 leaf-wise LGBM + 강한 제약 조합의 우위).
3. **V1-t 파라미터의 수렴 확인** — Optuna 60 trials(M6.A6)와 별개의 36조합 그리드가 −0.2%까지밖에
   못 접근 → 채택 파라미터가 우연이 아니라 최적 근방. "과적합 아닌가" 계열 질문의 마지막 조각.
4. **튜닝 자체는 중요** — 그리드 내 분산이 큼(LGBM 49.2~54.6, sMAPE 5%p+). 다만 어떤 조합도
   가족 순위를 뒤집지 못함. 90개 조합 전부 MLflow `sajura_grid_search`에 기록
   (UI: `cd AI && mlflow ui --backend-store-uri sqlite:///mlruns/mlflow.db`).

In [ ]:
import itertools
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
import mlflow
from catboost import CatBoostRegressor
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")
_here = Path.cwd()
AI_DIR = next(p for p in [_here.parent, _here, _here / "AI"] if (p / "data_prep").exists())
sys.path.insert(0, str(AI_DIR / "data_prep"))
import preprocess as pp

mlflow.set_tracking_uri(f"sqlite:///{AI_DIR}/mlruns/mlflow.db")
experiment_name = "sajura_grid_search"
mlflow.set_experiment(experiment_name)

# 재실행 시 이전 run 정리(리더보드 중복 방지 — soft delete)
_client = mlflow.tracking.MlflowClient()
_exp = _client.get_experiment_by_name(experiment_name)
for _r in _client.search_runs([_exp.experiment_id]):
    _client.delete_run(_r.info.run_id)

feat = pd.read_parquet(AI_DIR / "data/processed/features_daily.parquet").set_index("date")
feat, _ = pp.impute_weather(feat)
ob = feat[feat.is_open].copy()
y, tx = ob.total_amount, ob.tx_count
folds = pp.make_monthly_folds(ob.index)
SEL = folds[:-1]
print("선택 fold:", [f["month"] for f in SEL], "| test 봉인:", folds[-1]["month"])

STATIC = ["is_holiday", "semester_week", "is_semester_first2w", "temp_avg", "temp_range",
          "is_post_renewal", "days_since_reopen"]
s = y.copy()
X1 = ob[STATIC].copy()
X1["lag_sales_h"] = s.shift(1)
X1["lag_tx_h"] = tx.shift(1)
r7 = s.shift(1).rolling(7).mean()
X1["roll7_h"] = r7
X1["roll_atv_h"] = (s / tx).shift(1).rolling(7).mean()
bydow = s.groupby(s.index.dayofweek)
X1["lag_dow"] = bydow.shift(1)
X1["roll4dow"] = bydow.apply(lambda g: g.shift(1).rolling(4).mean()).droplevel(0)
X1 = pd.concat([X1, pd.get_dummies(ob.index.dayofweek, prefix="dow").set_index(ob.index)], axis=1)
b = X1.select_dtypes(bool).columns
X1[b] = X1[b].astype(int)
y_ratio = np.log1p(y) - np.log1p(r7)
XF = X1.astype(float)  # xgb·catboost용 — NaN은 양쪽 다 네이티브 처리


def mae(a, p):
    return float(np.mean(np.abs(np.asarray(a) - np.asarray(p))))


def smape(a, p):
    a, p = np.asarray(a), np.asarray(p)
    return float(np.mean(2 * np.abs(a - p) / (a + np.abs(p))) * 100)


NAIVE_MAE = float(np.mean([mae(y.loc[f["val"]], r7.loc[f["val"]]) for f in SEL]))
V1T_BEST = dict(learning_rate=0.0257, num_leaves=9, min_child_samples=10, subsample=0.7093,
                colsample_bytree=0.6796, reg_alpha=0.001, reg_lambda=0.0)


def eval_config(family, params):
    """5 fold walk-forward — 기존 프로토콜(800 + 조기 종료 50)."""
    maes, smapes = [], []
    for f in SEL:
        tr, va = f["train"], f["val"]
        ytr, yva = y_ratio.loc[tr], y_ratio.loc[va]
        k = ytr.notna()
        if family == "lgbm":
            m = lgb.LGBMRegressor(n_estimators=800, random_state=42, verbosity=-1,
                                  objective="l2", **params)
            m.fit(X1.loc[tr][k], ytr[k], eval_set=[(X1.loc[va], yva)],
                  callbacks=[lgb.early_stopping(50, verbose=False)])
            p = m.predict(X1.loc[va])
        elif family == "xgb":
            m = XGBRegressor(n_estimators=800, random_state=42, verbosity=0,
                             early_stopping_rounds=50, **params)
            m.fit(XF.loc[tr][k], ytr[k], eval_set=[(XF.loc[va], yva)], verbose=False)
            p = m.predict(XF.loc[va])
        else:
            m = CatBoostRegressor(iterations=800, random_seed=42, verbose=False,
                                  allow_writing_files=False, od_type="Iter", od_wait=50, **params)
            m.fit(XF.loc[tr][k], ytr[k], eval_set=(XF.loc[va], yva))
            p = m.predict(XF.loc[va])
        pred = np.expm1(p + np.log1p(r7.loc[va]))
        maes.append(mae(y.loc[va], pred))
        smapes.append(smape(y.loc[va], pred))
    return float(np.mean(maes)), float(np.mean(smapes))

In [ ]:
# ── 사전 선언 그리드 + V1-t 기준 run ──
GRIDS = {
    "lgbm": [dict(learning_rate=lr, num_leaves=nl, min_child_samples=mc, colsample_bytree=cs)
             for lr, nl, mc, cs in itertools.product([0.01, 0.03, 0.1], [7, 15, 31],
                                                     [5, 10], [0.7, 1.0])],
    "xgb": [dict(learning_rate=lr, max_depth=md, min_child_weight=mw, colsample_bytree=cs)
            for lr, md, mw, cs in itertools.product([0.01, 0.03, 0.1], [2, 3, 4],
                                                    [1, 5], [0.7, 1.0])],
    "cat": [dict(learning_rate=lr, depth=dp, l2_leaf_reg=l2)
            for lr, dp, l2 in itertools.product([0.01, 0.03, 0.1], [2, 4, 6], [1, 5])],
}
print("그리드 크기:", {k: len(v) for k, v in GRIDS.items()},
      "| 총 fit:", sum(len(v) for v in GRIDS.values()) * len(SEL))

v1t_mae, v1t_smape = eval_config("lgbm", V1T_BEST)
with mlflow.start_run(run_name="v1t_reference", tags={"family": "lgbm", "step": "reference"}):
    mlflow.log_params({**V1T_BEST, "note": "V1-t 채택값(Optuna 60 trials, M6.A6)"})
    mlflow.log_metrics({"smape_mean": v1t_smape,
                        "rel_mae_vs_ma7_pct": (v1t_mae / NAIVE_MAE - 1) * 100})
print(f"V1-t 기준: sMAPE {v1t_smape:.2f} (MA-7 대비 {(v1t_mae / NAIVE_MAE - 1) * 100:+.1f}%)")

In [ ]:
# ── 그리드 실행 — 세션2 [7] 패턴: 조합별 MLflow run ──
t0 = time.time()
results = []
for family, grid in GRIDS.items():
    for i, params in enumerate(grid, 1):
        m_mae, m_smape = eval_config(family, params)
        results.append(dict(family=family, mae=m_mae, smape=m_smape,
                            rel_v1t=(m_mae / v1t_mae - 1) * 100, **params))
        with mlflow.start_run(run_name=f"{family}_{i:02d}",
                              tags={"family": family, "step": "grid"}):
            mlflow.log_params(params)
            mlflow.log_metrics({"smape_mean": m_smape,
                                "rel_mae_vs_ma7_pct": (m_mae / NAIVE_MAE - 1) * 100,
                                "rel_mae_vs_v1t_pct": (m_mae / v1t_mae - 1) * 100})
    print(f"{family} {len(grid)}조합 완료 ({time.time()-t0:.0f}s)")
res = pd.DataFrame(results)

In [ ]:
# ── 비교·판정 — 세션2 [9][10] 패턴 search_runs 리더보드 + 사전 선언 판정 ──
experiment = mlflow.get_experiment_by_name(experiment_name)
runs_df = mlflow.search_runs(experiment_ids=[experiment.experiment_id])
columns_to_show = ["tags.mlflow.runName", "tags.family", "metrics.smape_mean",
                   "metrics.rel_mae_vs_v1t_pct"]
display(runs_df[columns_to_show].sort_values("metrics.smape_mean").head(10))

for family in GRIDS:
    g = res[res.family == family].sort_values("mae")
    pcols = [c for c in g.columns if c not in ("family", "mae", "smape", "rel_v1t")
             and g[c].notna().any()]
    print(f"\n[{family}] top-3:")
    display(g.head(3)[pcols + ["smape", "rel_v1t"]].round(3))

print("===== 판정 (사전 선언: 가족별 best가 V1-t 대비 MAE −5% 이상 개선 시 격상) =====")
for family in GRIDS:
    top = res[res.family == family].sort_values("mae").iloc[0]
    verdict = "격상" if top.rel_v1t <= -5 else (
        "재튜닝 검토 기록" if (family == "lgbm" and top.rel_v1t < -2) else "기각(기록만)")
    print(f"{family}: best vs V1-t {top.rel_v1t:+.1f}% → {verdict}")
print("\n가족별 그리드 내 sMAPE 분포(튜닝 민감도):")
display(res.groupby("family").smape.agg(["min", "median", "max"]).round(2))

### 관찰 — 세부

- **XGB·CatBoost 열세는 튜닝 부족이 아니었다**: 각자 36·18조합의 자기 그리드에서 best를 뽑아도
  V1-t 대비 +10~11%. 04 §3b(동일 설정 비교)의 결론이 "가족별 최적 설정 비교"에서도 유지 —
  **LightGBM 종결 판정을 재확인**. 소표본(영업일 ~250)에서는 잎 수를 강하게 묶은 leaf-wise
  LightGBM이 depth-wise 계열보다 유리한 구조.
- **V1-t 채택 파라미터의 강건성**: 독립적인 탐색 방법(그리드)이 Optuna 채택값의 −0.2% 이내로
  수렴 — 파라미터가 특정 탐색 경로의 요행이 아님. LGBM best(lr 0.1·leaves 31)와 채택값
  (lr 0.026·leaves 9)의 성능이 동률인 것은 이 체급에서 "정규화 총량"이 맞으면 세부 조합은
  둔감하다는 뜻.
- **주의(정직)**: 그리드 best는 검증 fold로 고른 값이라 그 자체가 낙관 편향 — 그래서 LGBM
  −0.2%는 "개선"이 아니라 "동률"로 읽는 것이 맞고, 사전 선언대로 채택값을 교체하지 않는다.
- 튜닝 민감도(가족 내 5%p+ 분산)는 발표에서 "왜 하이퍼파라미터 관리가 필요한가"의 근거 자료.

### 다음 단계

- spec 반영(docs PR): model_spec §3 초기 모델 항목에 "39차 그리드 재확인" 1줄.
- MLflow UI 스크린샷(91 runs 리더보드) — 발표용.